In [67]:
import cv2
from keras.models import load_model
from keras.preprocessing.image import load_img, img_to_array
import numpy as np
import tensorflow as tf
import keras
import time
import math

In [68]:
model = keras.models.load_model("asl_classifier.h5")

In [69]:
labels_dict = {0:'0', 
                 1:'A', 
                 2:'B', 
                 3:'C', 
                 4:'D', 
                 5:'E',
                 6:'F',
                 7:'G',
                 8:'H',
                 9:'I',
                 10:'J',
                 11:'K',
                 12:'L',
                 13:'M',
                 14:'N',
                 15:'O',
                 16:'P',
                 17:"Q",
                 18:'R',
                 19:'S',
                 20:'T', 
                 21:'U', 
                 22:'V',
                 23:'W',
                 24:'X',
                 25:'Y',
                 26:'Z'}
color_dict=(0,255,0)
x=0
y=0
w=64
h=64

# Fully Real-Time

In [70]:
img_size = 128
minValue = 70
source = cv2.VideoCapture(0)

# countdown configuration (seconds)
COUNTDOWN_SECONDS = 3.0
# how long to hold the "0" display / after-capture pause (seconds)
HOLD_ZERO_FOR = 0.2

start_time = time.perf_counter()
captured_this_cycle = False
last_capture_time = 0.0

# No accumulation string: show only the single letter captured per cycle
prev = " "  # currently-displayed letter for the last capture

while True:
    ret, img = source.read()
    if not ret:
        break

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    cv2.rectangle(img, (24, 24), (250, 250), color_dict, 2)
    crop_img = gray[24:250, 24:250]

    # Preprocess for model
    blur = cv2.GaussianBlur(crop_img, (5, 5), 2)
    th3 = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2
    )
    ret_t, res = cv2.threshold(
        th3, minValue, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    resized = cv2.resize(res, (img_size, img_size))
    normalized = resized / 255.0
    reshaped = np.reshape(normalized, (1, img_size, img_size, 1))
    result = model.predict(reshaped, verbose=0)
    label = np.argmax(result, axis=1)[0]

    # Time-based countdown
    now = time.perf_counter()
    elapsed = now - start_time

    if elapsed < COUNTDOWN_SECONDS:
        # show remaining seconds (3,2,1)
        remaining = COUNTDOWN_SECONDS - elapsed
        display_seconds = math.ceil(remaining)
        # keep prev showing from last capture until a new capture occurs
    else:
        # show 0 when elapsed >= duration
        display_seconds = 0
        if not captured_this_cycle:
            # Capture once at the end of the countdown
            # Instead of appending to a string, set prev to the single detected letter
            if label == 0:
                # keep convention: treat label 0 as a space/blank
                prev = " "
            else:
                prev = labels_dict.get(label, "")
            captured_this_cycle = True
            last_capture_time = now

    # After showing 0 for a small amount of time, reset the timer for the next countdown
    if captured_this_cycle and (now - last_capture_time) >= HOLD_ZERO_FOR:
        start_time = now
        captured_this_cycle = False
        # Do NOT append or accumulate; keep prev until replaced by the next capture
        # If you prefer to clear the displayed letter between cycles, uncomment the next line:
        # prev = " "

    # Draw texts
    # prev now holds exactly one letter (or a space) captured during the last completed countdown
    cv2.putText(img, prev, (24, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(img, str(display_seconds), (300, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 2)

    # Progress bar (optional): fraction of elapsed in the countdown
    fraction = min(1.0, max(0.0, elapsed / COUNTDOWN_SECONDS))
    # draw a simple horizontal progress bar at the bottom of the frame
    bar_x, bar_y, bar_w, bar_h = 24, img.shape[0] - 30, 226, 12
    cv2.rectangle(img, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (40, 40, 40), -1)
    cv2.rectangle(img, (bar_x, bar_y), (bar_x + int(bar_w * fraction), bar_y + bar_h), (0, 200, 0), -1)

    cv2.imshow("Gray", res)
    cv2.imshow("LIVE", img)
    key = cv2.waitKey(1)

    if key == 27:  # Esc to exit
        break

# Print the last captured single letter (instead of a built-up string)
print(prev)
cv2.destroyAllWindows()
source.release()
cv2.destroyAllWindows()

Q


In [71]:
# pip install gTTS

In [72]:
from gtts import gTTS 
  
# This module is imported so that we can  
# play the converted audio 
import os 
  
# The text that you want to convert to audio 
  
# Language in which you want to convert 
language = 'en'
# Passing the text and language to the engine,  
# here we have marked slow=False. Which tells  
# the module that the converted audio should  
# have a high speed 
myobj = gTTS(text=string, lang=language, slow=False) 
  
# Saving the converted audio in a mp3 file named 
# welcome  
myobj.save("welcome2121.mp3") 
  
# Playing the converted file 
os.system("welcome.mp3") 

PermissionError: [Errno 13] Permission denied: 'welcome2121.mp3'

In [ ]:
from playsound import playsound
playsound('welcome2121.mp3')